# AG News Classification: Three-Model Comparison

**Models:** Custom Transformer Encoder, BiLSTM, and pretrained DistilBERT  
**Dataset:** AG News (4 classes: World, Sports, Business, Sci/Tech)

This is a clean, modular rewrite. It trains the models and generates the evaluation plots in this notebook. **Training from a fresh initialization can produce different metrics** from an earlier run; the notebook does not hard-code previous accuracies. DistilBERT uses pretrained weights, while the custom Transformer and BiLSTM are initialized from scratch.

## 1. Install dependencies

In [ ]:
!pip -q install datasets transformers accelerate scikit-learn seaborn matplotlib pandas

## 2. Imports and reproducibility

In [ ]:
import os, re, time, random, copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 3. Load AG News

In [ ]:
raw = load_dataset("fancyzhx/ag_news")
label_names = raw["train"].features["label"].names
print(raw)
print("Labels:", label_names)

## 4. Shared settings and optional quick-run mode

For the final comparison, use the full dataset. Set `QUICK_RUN = True` only to check that the pipeline works; quick-run metrics are not comparable with full training.

In [ ]:
QUICK_RUN = False
BATCH_SIZE = 64 if torch.cuda.is_available() else 16
MAX_LEN = 128
EPOCHS_CUSTOM = 4
EPOCHS_DISTILBERT = 3
LR_CUSTOM = 3e-4
LR_DISTILBERT = 2e-5
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

train_hf = raw["train"]
test_hf = raw["test"]
if QUICK_RUN:
    train_hf = train_hf.shuffle(seed=SEED).select(range(6000))
    test_hf = test_hf.shuffle(seed=SEED).select(range(1500))
NUM_CLASSES = len(label_names)

## 5. Build a vocabulary and tokenize for the custom models

In [ ]:
PAD, UNK = "<pad>", "<unk>"
def basic_tokenize(text):
    return re.findall(r"[a-z0-9]+|[^\w\s]", text.lower())

from collections import Counter
counter = Counter()
for row in train_hf:
    counter.update(basic_tokenize(row["text"]))

VOCAB_SIZE = 30000
vocab = {PAD: 0, UNK: 1}
for token, _ in counter.most_common(VOCAB_SIZE - len(vocab)):
    vocab[token] = len(vocab)
print("Vocabulary size:", len(vocab))

def encode_text(text, max_len=MAX_LEN):
    ids = [vocab.get(tok, vocab[UNK]) for tok in basic_tokenize(text)[:max_len]]
    if not ids: ids = [vocab[UNK]]
    ids += [vocab[PAD]] * (max_len - len(ids))
    return ids

class AGNewsCustomDataset(Dataset):
    def __init__(self, hf_dataset):
        self.texts = hf_dataset["text"]
        self.labels = hf_dataset["label"]
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (torch.tensor(encode_text(self.texts[idx]), dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

custom_train_loader = DataLoader(AGNewsCustomDataset(train_hf), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
custom_test_loader = DataLoader(AGNewsCustomDataset(test_hf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

## 6. Shared training and evaluation functions

In [ ]:
def train_custom_model(model, train_loader, epochs, lr, name):
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": []}
    # Use a fixed validation split from the training data for monitoring.
    # The official test set is kept for final evaluation only.
    val_count = max(1, int(0.05 * len(train_loader.dataset)))
    train_count = len(train_loader.dataset) - val_count
    train_subset, val_subset = torch.utils.data.random_split(
        train_loader.dataset, [train_count, val_count],
        generator=torch.Generator().manual_seed(SEED)
    )
    train_dl = DataLoader(train_subset, batch_size=train_loader.batch_size, shuffle=True, num_workers=NUM_WORKERS)
    val_dl = DataLoader(val_subset, batch_size=train_loader.batch_size, shuffle=False, num_workers=NUM_WORKERS)
    best_loss, best_state = float("inf"), None
    start = time.time()
    for epoch in range(epochs):
        model.train(); total, count = 0.0, 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item() * y.size(0); count += y.size(0)
        train_loss = total / max(count, 1)
        model.eval(); total, count = 0.0, 0
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                loss = criterion(model(x), y)
                total += loss.item() * y.size(0); count += y.size(0)
        val_loss = total / max(count, 1)
        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        print(f"{name} | Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
    if best_state is not None: model.load_state_dict(best_state)
    return model, history, time.time() - start

@torch.no_grad()
def evaluate_custom_model(model, loader):
    model.eval()
    all_y, all_pred, total_loss, n = [], [], 0.0, 0
    criterion = nn.CrossEntropyLoss()
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0); n += y.size(0)
        all_y.extend(y.cpu().numpy()); all_pred.extend(logits.argmax(1).cpu().numpy())
    return np.array(all_y), np.array(all_pred), total_loss / max(n, 1)

## 7. Model 1 — Custom Transformer Encoder (random initialization)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, d_model=128, nhead=4, layers=2, ff_dim=256, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.positional = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)
    def forward(self, input_ids):
        pad_mask = input_ids.eq(0)
        x = self.positional(self.embedding(input_ids) * math.sqrt(self.embedding.embedding_dim))
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        valid = (~pad_mask).unsqueeze(-1).float()
        pooled = (x * valid).sum(1) / valid.sum(1).clamp(min=1)
        return self.classifier(self.dropout(pooled))

transformer_model = TransformerClassifier(len(vocab), NUM_CLASSES)
print("Transformer parameters:", sum(p.numel() for p in transformer_model.parameters()))
transformer_model, transformer_history, transformer_time = train_custom_model(
    transformer_model, custom_train_loader, EPOCHS_CUSTOM, LR_CUSTOM, "Transformer"
)
y_true_t, y_pred_t, transformer_test_loss = evaluate_custom_model(transformer_model, custom_test_loader)

## 8. Model 2 — BiLSTM (random initialization)

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim=128, hidden_dim=128, layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim, hidden_size=hidden_dim, num_layers=layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, input_ids):
        mask = input_ids.ne(0).unsqueeze(-1).float()
        embedded = self.embedding(input_ids)
        output, _ = self.lstm(embedded)
        pooled = (output * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.classifier(self.dropout(pooled))

bilstm_model = BiLSTMClassifier(len(vocab), NUM_CLASSES)
print("BiLSTM parameters:", sum(p.numel() for p in bilstm_model.parameters()))
bilstm_model, bilstm_history, bilstm_time = train_custom_model(
    bilstm_model, custom_train_loader, EPOCHS_CUSTOM, LR_CUSTOM, "BiLSTM"
)
y_true_b, y_pred_b, bilstm_test_loss = evaluate_custom_model(bilstm_model, custom_test_loader)

## 9. Model 3 — DistilBERT (pretrained Transformer fine-tuning)

In [ ]:
DISTILBERT_NAME = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_NAME)

def tokenize_batch(batch):
    return bert_tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

bert_train = train_hf.map(tokenize_batch, batched=True)
bert_test = test_hf.map(tokenize_batch, batched=True)
bert_train = bert_train.rename_column("label", "labels")
bert_test = bert_test.rename_column("label", "labels")
keep_cols = {"input_ids", "attention_mask", "labels"}
bert_train = bert_train.remove_columns([c for c in bert_train.column_names if c not in keep_cols])
bert_test = bert_test.remove_columns([c for c in bert_test.column_names if c not in keep_cols])
bert_train.set_format("torch")
bert_test.set_format("torch")
bert_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)
bert_train_loader = DataLoader(bert_train, batch_size=16, shuffle=True, collate_fn=bert_collator)
bert_test_loader = DataLoader(bert_test, batch_size=32, shuffle=False, collate_fn=bert_collator)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_NAME, num_labels=NUM_CLASSES
).to(DEVICE)
bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=LR_DISTILBERT)
bert_history = {"train_loss": [], "val_loss": []}
start = time.time()
for epoch in range(EPOCHS_DISTILBERT):
    bert_model.train(); total, n = 0.0, 0
    for batch in bert_train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        bert_optimizer.zero_grad(set_to_none=True)
        outputs = bert_model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        bert_optimizer.step()
        total += loss.item() * batch["labels"].size(0); n += batch["labels"].size(0)
    train_loss = total / max(n, 1)
    bert_history["train_loss"].append(train_loss)
    bert_history["val_loss"].append(np.nan)
    print(f"DistilBERT | Epoch {epoch+1}/{EPOCHS_DISTILBERT} | train_loss={train_loss:.4f}")
distilbert_time = time.time() - start

bert_model.eval()
y_true_d, y_pred_d, bert_total_loss, bert_n = [], [], 0.0, 0
with torch.no_grad():
    for batch in bert_test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = bert_model(**batch)
        bert_total_loss += outputs.loss.item() * batch["labels"].size(0)
        bert_n += batch["labels"].size(0)
        y_true_d.extend(batch["labels"].cpu().numpy())
        y_pred_d.extend(outputs.logits.argmax(1).cpu().numpy())
y_true_d, y_pred_d = np.array(y_true_d), np.array(y_pred_d)
distilbert_test_loss = bert_total_loss / max(bert_n, 1)

## 10. Compare all models

In [ ]:
results = []
def add_result(name, y_true, y_pred, loss, elapsed, model):
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Test Loss": loss,
        "Training Time (min)": elapsed / 60,
        "Parameters": sum(p.numel() for p in model.parameters())
    })

add_result("Transformer Encoder", y_true_t, y_pred_t, transformer_test_loss, transformer_time, transformer_model)
add_result("BiLSTM", y_true_b, y_pred_b, bilstm_test_loss, bilstm_time, bilstm_model)
add_result("DistilBERT", y_true_d, y_pred_d, distilbert_test_loss, distilbert_time, bert_model)
results_df = pd.DataFrame(results)
display(results_df.style.format({
    "Accuracy": "{:.4%}", "Macro F1": "{:.4f}", "Test Loss": "{:.4f}",
    "Training Time (min)": "{:.2f}", "Parameters": "{:,}"
}))

## 11. Accuracy and Macro F1 comparison

In [ ]:
plot_df = results_df.melt(id_vars="Model", value_vars=["Accuracy", "Macro F1"], var_name="Metric", value_name="Score")
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
plt.ylim(0, 1)
plt.title("AG News: Accuracy and Macro F1 by Model")
plt.ylabel("Score")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 12. Confusion matrices

In [ ]:
predictions = {
    "Transformer Encoder": (y_true_t, y_pred_t),
    "BiLSTM": (y_true_b, y_pred_b),
    "DistilBERT": (y_true_d, y_pred_d)
}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, (yt, yp)) in zip(axes, predictions.items()):
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names,
                yticklabels=label_names, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.tick_params(axis="x", rotation=35)
    ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

## 13. Classification reports

In [ ]:
for name, (yt, yp) in predictions.items():
    print("\\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(classification_report(yt, yp, target_names=label_names, digits=4))

## 14. Training-loss curves

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, len(transformer_history["train_loss"])+1), transformer_history["train_loss"], marker="o", label="Transformer train")
plt.plot(range(1, len(transformer_history["val_loss"])+1), transformer_history["val_loss"], marker="o", linestyle="--", label="Transformer validation")
plt.plot(range(1, len(bilstm_history["train_loss"])+1), bilstm_history["train_loss"], marker="o", label="BiLSTM train")
plt.plot(range(1, len(bilstm_history["val_loss"])+1), bilstm_history["val_loss"], marker="o", linestyle="--", label="BiLSTM validation")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss")
plt.title("Custom Models: Training and Validation Loss")
plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(bert_history["train_loss"])+1), bert_history["train_loss"], marker="o", label="DistilBERT train")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss")
plt.title("DistilBERT Training Loss")
plt.legend(); plt.tight_layout(); plt.show()

## 15. Save model weights and result table

In [ ]:
os.makedirs("ag_news_outputs", exist_ok=True)
torch.save(transformer_model.state_dict(), "ag_news_outputs/transformer_encoder.pt")
torch.save(bilstm_model.state_dict(), "ag_news_outputs/bilstm.pt")
torch.save(bert_model.state_dict(), "ag_news_outputs/distilbert_finetuned.pt")
results_df.to_csv("ag_news_outputs/model_comparison.csv", index=False)
print("Saved models and comparison table to ag_news_outputs/")